# L19 · 大模型通识：Transformer 是什么

**学习目标**
- 用大白话理解「Token」「Embedding」「Attention（注意力）」
- 亲手实现一个「迷你注意力」并可视化，破除神秘感
- 建立「大模型 = 预测下一个词的概率机器」的心智

**前置依赖**：L07 numpy、L11 第一性原理  
**预计时长**：50 分钟  
**技术栈**：`numpy`、`matplotlib`（离线可运行，无需 API key）

---

## 概念讲解：大模型 = 超级「下一个字预测器」

ChatGPT 的本质很简单：**给你前文，它猜「下一个字最可能是什么」**，然后把猜的字接上，再猜下一个……循环成句。

支撑它的三大积木：
1. **Token**：文字被切成小块（词/字/子词），模型只认 Token
2. **Embedding**：每个 Token 变成一个数字向量（语义变成坐标）
3. **Attention（注意力）**：每个词「看」其他词时，给重要的词更高权重

本课我们亲手算一遍「注意力」，看它到底在干什么。

## 第一步：把句子变成向量（模拟 Embedding）

In [ ]:
import numpy as np
np.random.seed(0)
words = ["猫", "坐", "在", "垫子", "上"]
# 每个词用一个 4 维向量表示（真实模型是几千维）
embed = {w: np.random.randn(4) for w in words}
for w in words:
    print(f"  {w} → 向量 {np.round(embed[w], 2)}")

## 第二步：算「注意力」—— 谁更关注谁

In [ ]:
def attention_matrix(embeddings):
    # 相似度 = 向量点积；softmax 归一化成概率
    E = np.array(embeddings)
    scores = E @ E.T
    e = np.exp(scores - scores.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

A = attention_matrix([embed[w] for w in words])
print("注意力权重（行→列，每行和为1）：")
for i, w in enumerate(words):
    print(f"  {w}: " + "  ".join(f"{words[j]}{A[i,j]:.2f}" for j in range(len(words))))

# 🎯 AHA 顿悟单元格：看「注意力」在句子里怎么流动

运行下面代码。你会看到一张**热力图**：展示句子里每个词「关注」其他词的程度。
对角线亮表示「关注自己」，而「垫子」和「上」之间若出现高亮，就说明模型学会了「搭配关系」。

> 你刚刚手算的就是 Transformer 的核心——Attention。GPT 比你多亿万个参数、读了万亿字，
> 但「注意力」这块积木，和你在图里看到的完全一样。

In [ ]:
# ===== 运行我！看注意力热力图 =====
import numpy as np, matplotlib.pyplot as plt
np.random.seed(2)
words = ["猫", "坐", "在", "软垫", "上", "打盹"]
E = np.random.randn(len(words), 5)
# 人为让「软垫-上」「猫-打盹」更相关，模拟学过的语义
E[3] += E[4]*1.5; E[0] += E[5]*1.2
scores = E @ E.T
e = np.exp(scores - scores.max(axis=1, keepdims=True))
A = e / e.sum(axis=1, keepdims=True)

plt.figure(figsize=(6, 5))
plt.imshow(A, cmap="YlOrRd")
plt.xticks(range(len(words)), words, rotation=45)
plt.yticks(range(len(words)), words)
for i in range(len(words)):
    for j in range(len(words)):
        plt.text(j, i, f"{A[i,j]:.2f}", ha="center", va="center", fontsize=8)
plt.title("🔥 注意力热力图：每个词如何关注其他词")
plt.colorbar()
plt.tight_layout(); plt.show()
print("  🧠 看懂这张图，你就懂了 Transformer 的心脏。")

# 📝 讲师备课笔记（接手 Agent 专用）

**本课难点**：Attention 数学（点积+softmax）；向量相似度=语义接近。  
**易错点**：softmax 数值溢出（已减 max 防溢出）；中文显示（emoji 规避字体）。  
**AHA 机制**：手写注意力+热力图，把「黑盒」变「白盒」，强破神秘感，是阶段四情绪锚点。  
**离线策略**：完全 numpy，无 API key，保证可运行。  
**衔接**：L20 Prompt 工程（怎么跟这机器说话）；L24 微调（改它的参数）；L31+ 后训练。  
**依赖**：`pip install numpy matplotlib`。

# 📚 作业 / 下一步

1. 把句子换成你喜欢的，看注意力如何变化。
2. 搜索「GPT 是自回归模型」加深理解。
3. 下一课 **L20 Prompt 工程：和 AI 对话的艺术** —— 学会用「提示词」指挥大模型，无需训练也能千变万化。